# Classificação de imagens com CNN (STL-10) em PyTorch

Este notebook treina uma **rede neural convolucional (CNN)** simples para
classificar imagens do dataset **STL-10** (10 classes: airplane, bird, car,
cat, deer, dog, horse, monkey, ship, truck).

O notebook está dividido em duas versões, que fazem exatamente a mesma coisa,
mas em **hardwares diferentes**:

1. **Versão GPU/CPU** — usa `torch.device("cuda")` (ou CPU, se não houver GPU).
2. **Versão TPU (PyTorch/XLA)** — usa `torch_xla` para rodar em TPU no Colab.

> No Colab, escolha o runtime adequado antes de rodar cada versão:
> `Runtime > Change runtime type` → GPU (para a versão 1) ou TPU (para a versão 2).

---
## Parte 1 — Versão GPU / CPU

Esta versão detecta automaticamente se há uma GPU disponível e usa CUDA;
caso contrário, roda em CPU.

### 1.1 Importações e configuração do device

Importamos as bibliotecas principais:
- `torch`, `torch.nn`, `torch.optim` → construção e treino do modelo;
- `torchvision.datasets` / `transforms` → carregamento e pré-processamento do STL-10;
- `matplotlib` → visualização de imagens e curvas de treino.

Também definimos o `device` (GPU se disponível, senão CPU) e os nomes das
10 classes do STL-10, na mesma ordem usada pelos rótulos numéricos do dataset.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_names = [
    "airplane", "bird", "car", "cat", "deer",
    "dog", "horse", "monkey", "ship", "truck",
]

### 1.2 Pré-processamento e carregamento do dataset

Definimos as transformações aplicadas a cada imagem:
- `ToTensor()` converte a imagem PIL em tensor `(C, H, W)` com valores em `[0, 1]`;
- `Normalize(mean=0.5, std=0.5)` reescala os valores para o intervalo `[-1, 1]`,
  o que geralmente ajuda a estabilizar o treinamento.

Em seguida, baixamos e carregamos o **STL-10** (splits `train` e `test`) e
criamos os `DataLoader`s, responsáveis por gerar os batches de treino/teste.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_dataset = datasets.STL10(root="./data", split="train", download=True, transform=transform)
test_dataset = datasets.STL10(root="./data", split="test", download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

### 1.3 Definição da arquitetura da CNN

`SimpleCNN` é uma rede convolucional pequena composta por:

- **Bloco de features**: 3 camadas `Conv2d + ReLU + MaxPool2d`, que extraem
  características visuais e reduzem progressivamente a resolução espacial
  (32 → 64 → 128 canais);
- **Bloco classificador**: um `Flatten` seguido de duas camadas lineares
  (`128*12*12 → 256 → num_classes`), que produzem os logits finais para
  cada uma das 10 classes.

In [ ]:
class SimpleCNN(nn.Module):
    """A small convolutional network for STL-10 classification."""

    def __init__(self, num_classes: int = 10) -> None:
        """Initialize the layers of the network.

        Args:
            num_classes: Number of output classes.
        """
        super().__init__()
        self.features = nn.Sequential(
            ...
        )
        self.classifier = nn.Sequential(
            ...
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run a forward pass through the network.

        Args:
            x: Input batch of images.

        Returns:
            Class logits for each input image.
        """
        ...
        return x

### 1.4 Funções utilitárias de visualização

Essas funções auxiliam a **inspecionar os dados e os resultados**:

- `unnormalize`: desfaz a normalização aplicada antes, para exibir a imagem
  com cores corretas (`[0, 1]`);
- `show_initial_data`: mostra uma amostra de imagens do dataset antes do
  treino, com seus rótulos verdadeiros;
- `show_predictions`: mostra imagens de teste com a predição do modelo
  (em verde se correta, em vermelho se incorreta) após o treino;
- `show_loss_curve`: plota a curva de perda (loss) por época, para
  acompanhar a convergência do treinamento.

In [ ]:
def unnormalize(image: torch.Tensor) -> torch.Tensor:
    """Revert normalization for display purposes.

    Args:
        image: Normalized image tensor with shape (C, H, W).

    Returns:
        Image tensor rescaled back to the [0, 1] range.
    """
    mean = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)
    std = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)
    return image * std + mean


def show_initial_data(loader: DataLoader, num_images: int = 8) -> None:
    """Display a grid of sample images from a dataset before training.

    Args:
        loader: DataLoader providing the images to display.
        num_images: Number of images to display.
    """
    images, labels = next(iter(loader))
    images = images[:num_images]
    labels = labels[:num_images]

    fig, axes = plt.subplots(1, num_images, figsize=(num_images * 2, 2))
    for i, ax in enumerate(axes):
        image = unnormalize(images[i]).permute(1, 2, 0).clamp(0, 1)
        ax.imshow(image)
        ax.set_title(class_names[labels[i]])
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_predictions(model: nn.Module, loader: DataLoader, num_images: int = 8) -> None:
    """Display sample images with predicted and true labels after training.

    Args:
        model: The trained neural network.
        loader: DataLoader providing the images to display.
        num_images: Number of images to display.
    """
    model.eval()
    images, labels = next(iter(loader))
    images_device = images[:num_images].to(device)

    with torch.no_grad():
        outputs = model(images_device)
        predictions = outputs.argmax(dim=1).cpu()

    fig, axes = plt.subplots(1, num_images, figsize=(num_images * 2, 2))
    for i, ax in enumerate(axes):
        image = unnormalize(images[i]).permute(1, 2, 0).clamp(0, 1)
        ax.imshow(image)
        predicted_name = class_names[predictions[i]]
        true_name = class_names[labels[i]]
        color = "green" if predictions[i] == labels[i] else "red"
        ax.set_title(f"pred: {predicted_name}\ntrue: {true_name}", color=color, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_loss_curve(loss_history: list[float]) -> None:
    """Plot the training loss curve across epochs.

    Args:
        loss_history: List of average loss values, one per epoch.
    """
    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("training loss convergence")
    plt.grid(True)
    plt.show()

### 1.5 Funções de treino e avaliação

- `train_one_epoch`: executa uma época de treino — forward, cálculo da loss
  (`CrossEntropyLoss`), backward e atualização dos pesos via `optimizer.step()` —
  e retorna a loss média da época;
- `evaluate`: percorre o `DataLoader` de teste em modo de avaliação
  (`model.eval()`, sem gradientes) e retorna a acurácia.

In [ ]:
def train_one_epoch(model: nn.Module, loader: DataLoader) -> float:
    """Train the model for a single epoch.

    Args:
        model: The neural network to train.
        loader: DataLoader providing training batches.

    Returns:
        Average training loss for the epoch.
    """
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        ...

        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model: nn.Module, loader: DataLoader) -> float:
    """Evaluate the model's accuracy on a dataset.

    Args:
        model: The neural network to evaluate.
        loader: DataLoader providing evaluation batches.

    Returns:
        Accuracy as a value between 0 and 1.
    """
    model.eval()
    correct = 0
    total = 0
    ...
    return correct / total

### 1.6 Treinamento do modelo

1. Exibimos algumas imagens de treino antes de começar (`show_initial_data`);
2. Instanciamos o modelo, a função de perda (`CrossEntropyLoss`) e o
   otimizador (`Adam`, `lr=1e-3`);
3. Treinamos por `num_epochs` épocas, imprimindo a loss de treino e a
   acurácia no teste a cada época.

In [ ]:
show_initial_data(train_loader)

model = ...
criterion = ...
optimizer = ...

num_epochs = 10
loss_history = []

for epoch in range(num_epochs):
    ...
    print(f"Epoch {epoch + 1}/{num_epochs} - loss: {train_loss:.4f} - test_accuracy: {test_accuracy:.4f}")

### 1.7 Resultados

Por fim, visualizamos a **curva de loss** ao longo das épocas e algumas
**predições do modelo** no conjunto de teste (verde = correta, vermelho = errada).

In [ ]:
show_loss_curve(loss_history)
show_predictions(model, test_loader)

---
## Parte 2 — Versão TPU (PyTorch/XLA)

Esta versão é equivalente à Parte 1, mas roda em **TPU** usando o pacote
`torch_xla`. As principais diferenças estão marcadas nos comentários:

- o device é `xm.xla_device()` em vez de `torch.device("cuda")`;
- o passo de otimização usa `xm.optimizer_step(optimizer)` em vez de
  `optimizer.step()`;
- é necessário chamar `xm.mark_step()` para sincronizar a execução do grafo XLA.

> Selecione um runtime **TPU** no Colab antes de rodar esta parte
> (`Runtime > Change runtime type > TPU`).

### 2.1 Instalação do PyTorch/XLA

Instala o pacote `torch_xla` com suporte a TPU. Rode esta célula primeiro,
em um runtime TPU.

In [ ]:
# Install PyTorch/XLA for Colab (TPU runtime) — run this in a separate cell first
!uv pip install torch_xla[tpu]

### 2.2 Importações e configuração do device (TPU)

Mesma ideia da versão GPU/CPU, mas importando `torch_xla.core.xla_model`
e definindo o `device` como o dispositivo XLA (TPU).

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch_xla.core.xla_model as xm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = xm.xla_device()

class_names = [
    "airplane", "bird", "car", "cat", "deer",
    "dog", "horse", "monkey", "ship", "truck",
]

### 2.3 Pré-processamento e carregamento do dataset

Idêntico à versão GPU/CPU: normalização das imagens e criação dos
`DataLoader`s de treino e teste a partir do STL-10.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_dataset = datasets.STL10(root="./data", split="train", download=True, transform=transform)
test_dataset = datasets.STL10(root="./data", split="test", download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

### 2.4 Definição da arquitetura da CNN

Mesma arquitetura `SimpleCNN` usada na versão GPU/CPU (3 blocos
convolucionais + classificador linear).

In [ ]:
class SimpleCNN(nn.Module):
    """A small convolutional network for STL-10 classification."""

    def __init__(self, num_classes: int = 10) -> None:
        """Initialize the layers of the network.

        Args:
            num_classes: Number of output classes.
        """
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 12 * 12, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run a forward pass through the network.

        Args:
            x: Input batch of images.

        Returns:
            Class logits for each input image.
        """
        x = self.features(x)
        x = self.classifier(x)
        return x

### 2.5 Funções utilitárias de visualização

Mesmas funções da versão GPU/CPU, usadas para visualizar amostras do
dataset, predições do modelo e a curva de loss.

In [ ]:
def unnormalize(image: torch.Tensor) -> torch.Tensor:
    """Revert normalization for display purposes.

    Args:
        image: Normalized image tensor with shape (C, H, W).

    Returns:
        Image tensor rescaled back to the [0, 1] range.
    """
    mean = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)
    std = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)
    return image * std + mean


def show_initial_data(loader: DataLoader, num_images: int = 8) -> None:
    """Display a grid of sample images from a dataset before training.

    Args:
        loader: DataLoader providing the images to display.
        num_images: Number of images to display.
    """
    images, labels = next(iter(loader))
    images = images[:num_images]
    labels = labels[:num_images]

    fig, axes = plt.subplots(1, num_images, figsize=(num_images * 2, 2))
    for i, ax in enumerate(axes):
        image = unnormalize(images[i]).permute(1, 2, 0).clamp(0, 1)
        ax.imshow(image)
        ax.set_title(class_names[labels[i]])
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_predictions(model: nn.Module, loader: DataLoader, num_images: int = 8) -> None:
    """Display sample images with predicted and true labels after training.

    Args:
        model: The trained neural network.
        loader: DataLoader providing the images to display.
        num_images: Number of images to display.
    """
    model.eval()
    images, labels = next(iter(loader))
    images_device = images[:num_images].to(device)

    with torch.no_grad():
        outputs = model(images_device)
        predictions = outputs.argmax(dim=1).cpu()

    fig, axes = plt.subplots(1, num_images, figsize=(num_images * 2, 2))
    for i, ax in enumerate(axes):
        image = unnormalize(images[i]).permute(1, 2, 0).clamp(0, 1)
        ax.imshow(image)
        predicted_name = class_names[predictions[i]]
        true_name = class_names[labels[i]]
        color = "green" if predictions[i] == labels[i] else "red"
        ax.set_title(f"pred: {predicted_name}\ntrue: {true_name}", color=color, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_loss_curve(loss_history: list[float]) -> None:
    """Plot the training loss curve across epochs.

    Args:
        loss_history: List of average loss values, one per epoch.
    """
    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("training loss convergence")
    plt.grid(True)
    plt.show()

### 2.6 Funções de treino e avaliação (TPU)

Iguais à versão GPU/CPU, com duas diferenças específicas do XLA:

- `xm.optimizer_step(optimizer)` substitui `optimizer.step()` — ele já
  cuida de sincronizar os gradientes entre os núcleos da TPU antes de
  atualizar os pesos;
- `xm.mark_step()` sinaliza ao XLA que o grafo computacional acumulado até
  aquele ponto deve ser executado (o XLA constrói o grafo de forma lazy).

In [ ]:
def train_one_epoch(model: nn.Module, loader: DataLoader) -> float:
    """Train the model for a single epoch on TPU.

    Args:
        model: The neural network to train.
        loader: DataLoader providing training batches.

    Returns:
        Average training loss for the epoch.
    """
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        xm.optimizer_step(optimizer)
        xm.mark_step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model: nn.Module, loader: DataLoader) -> float:
    """Evaluate the model's accuracy on a dataset using TPU.

    Args:
        model: The neural network to evaluate.
        loader: DataLoader providing evaluation batches.

    Returns:
        Accuracy as a value between 0 and 1.
    """
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            xm.mark_step()
    return correct / total

### 2.7 Treinamento do modelo (TPU)

Mesmo fluxo da versão GPU/CPU: mostra amostras de treino, cria modelo/
loss/otimizador e treina por `num_epochs` épocas, agora executando na TPU.

In [ ]:
show_initial_data(train_loader)

model = SimpleCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10
loss_history = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader)
    test_accuracy = evaluate(model, test_loader)
    loss_history.append(train_loss)
    print(f"Epoch {epoch + 1}/{num_epochs} - loss: {train_loss:.4f} - test_accuracy: {test_accuracy:.4f}")

### 2.8 Resultados (TPU)

Curva de loss e predições do modelo treinado na TPU, no mesmo formato da
versão GPU/CPU.

In [ ]:
show_loss_curve(loss_history)
show_predictions(model, test_loader)